STREAM DATA INGESTION

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
import json

env = spark.conf.get(
    "ENV",
    "dev"
)
# /Repos/kishorecoder685@gmail.com/banking-fraud-data-platform/configs
config = json.load(
    open(f"/Workspace/Repos/myrepo@gmail.com/banking-fraud-data-platform/configs/{env}.json")
)

# Kafka Stream
kafka_df = spark.readStream \
    .format("kafka") \
    # Connect to Kafka Server
    .option("kafka.bootstraps.servers","localhost:9092") \ 
    # Reading transactions table, Streaming the event
    .option("subscribe", config["kafka_topic"]) \
    .option("startingOffsets","latest") \
    .load()

# Schema Enforcement
transaction_schema = StructType([
    StructField("transaction_id", StringType()),
    StructField("customer_id",StringType()),
    StructField("transaction_amount",DoubleType()),
    StructField("account_number",StringType()),
    StructField("transaction_timestamp", TimeStampType()),
    StructField("merchant_id", StringType()),
    StructField("merchant_category",StringType()),
    StructField("device_id",StringType()),
    StructField("location",StringType())
])

# getting the value from Kafka which is in JSON format and convert to dataframe (transaction_schema)
# Transaction schema has necessary columns and their datatypes for schema validation
parsed_transactions = kafka_df.select(
    from_json(
        col("value").cast("string"),transaction_schema).alias("data").select("data.*")
)

# Convert the Binary data to String, Kafka provides the data in binary format
df = kafka_df.selectExpr("CAST(value AS STRING)")

# Define the datatypes of the column
schema = "transaction_id STRING, user_id INT, amount DOUBLE, timestamp STRING"

df = df.select(F.from_json("value",schema).alias("data")).select("data.*")


# BAD RECORDS -- Adding DQL

bad_records = parsed_transactions.filter(
    col("transaction_id").isNull() |
    col("customer_id").isNUll() |
    col("transaction_amount").isNull()
)

(
    bad_records.select(to_json(struct("*")).alias("value"))
                .writeStream
                .format("kafka")
                .option("kafka.bootstrap.servers", "localhost:9092")
                .option("topic", "bad_records")
                .option("checkpointLocation","/chk/dlq")
                .start()
)

# Write the transaction table to Delta Bronze
df.writeStream.format("delta").output_mode("append").option("checkpointLocation",config["bronze_checkpoint"]).start(config["bronze_path"])


BATCH DATA INGESTION

In [0]:
customer_df = spark.read.format("jdbc")\
              .option("url", "jdbc:mysql://localhost:3306/bank") \
              .option("dbtable","customers") \
              .option("user","root") \
              .option("password","*********") \
              .load()

# Writing the customer table in delta format and this ensures the Shcema Enforcement. If in some cases if we don't need to enforce the schema, This is a questions?
customer_df.write.format("delta").mode("overwrite").saveAsTable("/delta/bronze/customers")